## Import packages

In [ ]:
import astropy
from pylab import *
from astropy.io import fits
from astropy.time import Time
from PyAstronomy import pyasl
from astropy import units as u
from astropy.coordinates import SkyCoord, EarthLocation


## Read the FITS file

## 1- In the following line, we will read the observational data from the .fits file given to you

In [ ]:
# §4.2.4: the TARGET here is G64-12, the radial-velocity standard - not our star. Same
# program, different input, which is the point of the exercise.
import numpy as np
wvl2, flx2 = (np.asarray(x, float) for x in pyasl.read1dFitsSpec("../../../data/G64-12.fits"))
print(f"target  G64-12   {len(wvl2)} px, {wvl2[0]:.0f}-{wvl2[-1]:.0f} A")

In [ ]:
wvl1, flx1 = pyasl.read1dFitsSpec("../../../data/HD140283.fits")   # template
# checked: already at rest (H-alpha core at 6562.77 A), so no pre-shift is needed


## Plot the observed spectrum and a comparison spectrum near the Mg b lines to take a look

In [ ]:
fig  = plt.figure(figsize=(12,7))
#plt.plot(wvl2, flx2, 'r.-',label="j0232-3708")
plt.plot(wvl2, flx2, 'r.-',label="G64-12")

#plt.plot(wvl1, flx1, 'b--',label="HD 122563")
plt.plot(wvl1, flx1, 'b--',label="HD140283")


# lines below are used to control the tick markers
plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
# lines below are used to add labels
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
# lines below are used to control the x- and y-axis limits. This wavelength range shows the Mg triplet lines.
xlim([5161,5190])
ylim([0.0,1.1])
# line below is used to show the legend
plt.legend(loc="lower right", fontsize=16)
# line below is used to show the figure
plt.show()


In [ ]:
#star
ind = np.where((6560.0 < wvl2) & (wvl2< 6575.0))
waveSpectrum = wvl2[ind[0]]
fluxSpectrum = flx2[ind[0]]

#template
indx= np.where((6530.0 < wvl1) & (wvl1< 6600.0))
waveTemplate = wvl1[indx[0]]
fluxTemplate = flx1[indx[0]]


## Plot the spectral region you want to run the cross-correlation on (= Halpha region) to check that everything looks alright

In [ ]:
# Plot the chosen wavelength regime. Recall that the comparison star spectrum is at rest, the observed one isn't (yet).
fig  = plt.figure(figsize=(12,7))
plt.plot(waveSpectrum, fluxSpectrum, 'r.-',label="G64-12")

#plt.plot(waveTemplate, fluxTemplate, 'b--',label="HD122563")
plt.plot(waveTemplate, fluxTemplate, 'b--',label="HD140283")

plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
xlim([6550,6580])
ylim([0.0,1.1])
plt.legend(loc="lower right", fontsize=16)
plt.show()


## Carry out the cross-correlation.

In [ ]:

# AF scans -10..+200 km/s, which does NOT contain our star at ~+350 (her comment assumes a
# shift under 3 A; ours is 6 A). Widened; everything else is her call, unchanged.
# Second change to her call: she correlates the RAW fluxes. Both spectra sit near a continuum
# of 1 with absorption dips, so sum(f*t) is LARGEST where the dips fail to overlap - the naive
# CCF has no real peak and argmax lands on the scan edge (verified: it returns rvmin exactly).
# Subtracting the continuum makes the lines the signal, which is what the method assumes.
fluxSpectrum_cs = fluxSpectrum - np.median(fluxSpectrum)
fluxTemplate_cs = fluxTemplate - np.median(fluxTemplate)
rv, cc = pyasl.crosscorrRV(waveSpectrum, fluxSpectrum_cs, waveTemplate, fluxTemplate_cs, -100., 500, 0.1, skipedge=20)



## Find the maximum the of cross-correlation function

In [ ]:
maxind = np.argmax(cc)

## Plot the cross-correlation result

In [ ]:
plt.plot(rv, cc, 'bp-')
plt.plot(rv[maxind], cc[maxind], 'ro')
plt.show()

## Print the value of RV

In [ ]:
print (round(rv[maxind], 2))  # just the plain number  --- USE FOR APPLY_RADVEL_VELOCITY

In [ ]:
print("G64-12 has a geocentric radial velocity = ", round(rv[maxind], 2), " km/s ")
print()
if rv[maxind] > 0.0:
    print("There is a red-shift with respect to the template")
else:
    print("There is a blue-shift with respect to the template")

print('Is the star receding or approaching us?')

### Save for the other notebooks

In [ ]:
import json, os
os.makedirs("results", exist_ok=True)
out = {"target": "G64-12.fits", "v_geo_kms": round(float(rv[maxind]), 2),
       "method": "pyasl.crosscorrRV, AF's calculate_geocentric_radial_velocity_correction_af.ipynb",
       "template": "HD140283.fits (already at rest)",
       "star_window_A": [6560.0, 6575.0], "template_window_A": [6530.0, 6600.0],
       "rv_scan_kms": [-100.0, 500.0, 0.1], "skipedge": 20,
       "_note": "AF's scan range -10..200 widened to reach +350; grid step 0.1 km/s sets the precision"}
json.dump(out, open("results/part4-2-4_g64-12_rv_HD140283.json","w"), indent=2)
print(json.dumps(out, indent=2))